# 05 — Feature-based model

Direct multi-horizon gradient boosting, with the leakage guarantee verified
inline.

**Report sections fed:** 7 (Feature-based model).


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)


In [ ]:
frame = data.load_hourly()
y = frame[config.TARGET]

y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"train {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)})")
print(f"test  {test_index.min()} -> {test_index.max()}  ({len(y_test)})")


## Design matrix

One row per *(target timestamp, horizon)* pair. Target lags are anchored to the
forecast origin, so `lag_k = y.shift(h - 1 + k)`: the shift scales with the
horizon and no feature can reach inside the forecast window.


In [ ]:
indoor = [c for c in config.INDOOR_COLS if c in frame.columns]
weather = [c for c in config.WEATHER_COLS if c in frame.columns]

design, feature_cols = feature_models.build_design(
    y, exog_origin=frame[indoor + weather], exog_future=None,
    max_horizon=config.HORIZON,
)

print(f"{design.shape[0]} rows x {len(feature_cols)} features")
design.head()


### Leakage check\n\nPerturb every test observation; no training-row feature may respond.

In [ ]:
tampered = y.copy()
tampered.loc[test_index] += 10_000

alt, _ = feature_models.build_design(
    tampered, exog_origin=frame[indoor + weather], exog_future=None,
    max_horizon=config.HORIZON,
)

train_mask = design.index < test_index[0]
cols = [c for c in feature_cols if c != "horizon"]

print("training features unchanged:",
      np.allclose(design.loc[train_mask, cols], alt.loc[train_mask, cols]))


## Fit

In [ ]:
train_rows = design.loc[design.index < test_index[0]]
X_train, y_train_ml = train_rows[feature_cols], train_rows["target"]

model = feature_models.fit_feature_model(X_train, y_train_ml)
print(f"n_iter_ = {model.n_iter_} (of max_iter; equality means early stopping never fired)")


### Optional: tune under blocked time-series CV\n\nSelection confined to the training sample.

In [ ]:
# best = feature_models.tune_feature_model(X_train, y_train_ml)
# print(best)


## Rolling-origin prediction

In [ ]:
pred = feature_models.rolling_origin_predict(
    model, design, feature_cols, test_index, config.HORIZON)

evaluation.evaluate_all({"feature_model": pred}, y_test, y_train).round(3)


## Permutation importance

Preferred to split-count importance because nine indoor temperature sensors in
one dwelling are close to collinear, and split counts divide arbitrarily among
collinear features.


In [ ]:
test_rows = features.select_rows(
    design, features.rolling_origin_pairs(test_index, config.HORIZON))

importance = feature_models.feature_importance(
    model, test_rows[feature_cols], test_rows["target"])

importance.head(20)


In [ ]:
fig = plotting.plot_feature_importance(importance)


### Do the lagged indoor sensors contribute anything?\n\nDirect evidence for report Section 7.3.

In [ ]:
sensor_features = [f for f in importance["feature"] if f.endswith("_origin")]
share = importance.set_index("feature").loc[sensor_features, "importance"].sum()
total = importance["importance"].clip(lower=0).sum()
print(f"lagged exogenous share of total importance: {share / total:.1%}")
